# Flickr30k T3→S3 INT8 Non-Inferiority

Phân tích paired bootstrap theo ảnh sau khi ứng viên mixed INT8 vượt toàn bộ
metric chất lượng nhưng chỉ đạt 75,64% caption trùng nguyên văn, thấp hơn
strict-gate 90%. Notebook **không hạ ngưỡng**, không sinh caption lại và không
đọc Held-out Test.

## Kaggle input bắt buộc — đúng 2 input

1. Output `F30K_T3S3_INT8_Validation` đã sinh đủ 1.014 caption.
2. Package/dataset `pycocoevalcap-1.2`.

Runtime: CPU / Accelerator None / Internet Off, dự kiến 2–5 phút.

Endpoint chính: delta CIDEr-D theo từng `image_id`,
`mixed INT8 − ONNX FP32`. Thực hiện 10.000 paired bootstrap replicate với seed
20261026. Biên không-kém-hơn là **−0,010 CIDEr-D**, đúng bằng mức giảm tối đa
đã khóa trong notebook Validation trước khi xem kết quả. Kết luận đạt khi cận
dưới khoảng tin cậy percentile 95% lớn hơn −0,010.

Phân tích `CI > 0` chỉ mang tính thăm dò về ưu thế; không được dùng để tuyên bố
INT8 tốt hơn một cách xác nhận. Kết quả strict verbatim-agreement 90% luôn được
giữ là thất bại trong báo cáo.

## Cell 1 — Authorization and preservation of the failed strict gate

In [1]:
import hashlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

EXPERIMENT_ID = "F30K_T3S3_INT8_NonInferiority"
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working") / EXPERIMENT_ID
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json_atomic(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def unique_named(name):
    matches = sorted(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    assert len(matches) == 1, (f"Cần đúng một {name}", [str(x) for x in matches])
    return matches[0]

VALIDATION_RESULT_PATH = unique_named(
    "F30K_T3S3_INT8_Validation_RESULT.json"
)
CAPTIONS_PATH = unique_named(
    "F30K_T3S3_INT8_Validation_CAPTIONS.csv"
)
VALIDATION_RESULT = read_json(VALIDATION_RESULT_PATH)

assert VALIDATION_RESULT["experiment_id"] == (
    "F30K_T3S3_INT8_Validation"
)
assert VALIDATION_RESULT["validation_images"] == 1014
assert VALIDATION_RESULT["validation_references"] == 5070
assert VALIDATION_RESULT["evaluation_split"] == "Flickr30k Karpathy Validation"
assert VALIDATION_RESULT["test_set_status"] == "sealed"
assert VALIDATION_RESULT["test_set_used"] is False

# Chấp nhận output v1 đã hoàn thành inference nhưng bị assert do strict
# verbatim-agreement gate. Mọi gate và số liệu gốc vẫn phải còn nguyên.
original_gates = VALIDATION_RESULT["gates"]
assert all(
    record["passed"]
    for record in original_gates["fp32_reproduction"].values()
)
assert all(
    record["passed"]
    for record in original_gates["mixed_int8_quality"].values()
)
assert original_gates["size"]["passed"] is True
assert original_gates["caption_agreement"]["passed"] is False
assert original_gates["caption_agreement"]["minimum_required"] == 0.90
assert abs(
    original_gates["caption_agreement"]["agreement"]
    - 0.7564102564102564
) < 1e-12

if importlib.util.find_spec("pycocoevalcap") is None:
    wheels = sorted(INPUT_ROOT.rglob("pycocoevalcap*.whl"))
    assert wheels, "Hãy gắn package/dataset pycocoevalcap-1.2 offline."
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", str(wheels[0])]
    )
assert importlib.util.find_spec("pycocoevalcap") is not None

print("Authorization           : PASSED")
print("Validation source       :", VALIDATION_RESULT["status"])
print("Scientific metric gates : PASSED")
print("Original strict gate    : FAILED / 75.641% < 90%")
print("Held-out Test           : SEALED / NOT READ")

Authorization           : PASSED
Validation source       : passed_with_caption_variation_review_required
Scientific metric gates : PASSED
Original strict gate    : FAILED / 75.641% < 90%
Held-out Test           : SEALED / NOT READ


## Cell 2 — Official per-image CIDEr-D reconstruction

In [2]:
captions = pd.read_csv(CAPTIONS_PATH)
expected_columns = {
    "image_id",
    "fp32_caption",
    "mixed_int8_caption",
    "exact_match",
    "reference_1",
    "reference_2",
    "reference_3",
    "reference_4",
    "reference_5",
}
assert set(captions.columns) == expected_columns
assert len(captions) == 1014
assert captions["image_id"].nunique() == 1014
assert captions["image_id"].is_monotonic_increasing
assert int(captions["exact_match"].sum()) == 767
assert abs(float(captions["exact_match"].mean()) - 0.7564102564102564) < 1e-12
assert not captions[list(expected_columns - {"image_id", "exact_match"})].isna().any().any()

from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.cider.cider import Cider

image_ids = captions["image_id"].astype(int).tolist()
references = {}
fp32_hypotheses = {}
int8_hypotheses = {}
for row in captions.itertuples(index=False):
    image_id = int(row.image_id)
    references[image_id] = [
        {"caption": getattr(row, f"reference_{index}")}
        for index in range(1, 6)
    ]
    fp32_hypotheses[image_id] = [{"caption": row.fp32_caption}]
    int8_hypotheses[image_id] = [{"caption": row.mixed_int8_caption}]

tokenizer = PTBTokenizer()
tokenized_references = tokenizer.tokenize(references)
tokenized_fp32 = tokenizer.tokenize(fp32_hypotheses)
tokenized_int8 = tokenizer.tokenize(int8_hypotheses)

cider = Cider()
fp32_mean, fp32_per_image = cider.compute_score(
    tokenized_references, tokenized_fp32
)
int8_mean, int8_per_image = cider.compute_score(
    tokenized_references, tokenized_int8
)
fp32_per_image = np.asarray(fp32_per_image, dtype=np.float64)
int8_per_image = np.asarray(int8_per_image, dtype=np.float64)
assert fp32_per_image.shape == (1014,)
assert int8_per_image.shape == (1014,)
assert np.isfinite(fp32_per_image).all()
assert np.isfinite(int8_per_image).all()

reported_fp32 = VALIDATION_RESULT["metrics"]["onnx_fp32"]["CIDEr_D"]
reported_int8 = VALIDATION_RESULT["metrics"]["mixed_int8"]["CIDEr_D"]
assert abs(float(fp32_mean) - reported_fp32) < 1e-12
assert abs(float(int8_mean) - reported_int8) < 1e-12

paired_delta = int8_per_image - fp32_per_image
observed_delta = float(paired_delta.mean())
assert abs(
    observed_delta
    - VALIDATION_RESULT["metrics"]["mixed_minus_fp32"]["CIDEr_D"]
) < 1e-12

per_image = pd.DataFrame({
    "image_id": image_ids,
    "fp32_CIDEr_D": fp32_per_image,
    "mixed_int8_CIDEr_D": int8_per_image,
    "delta_CIDEr_D": paired_delta,
    "exact_caption_match": captions["exact_match"].astype(bool),
})
per_image_path = WORK_ROOT / f"{EXPERIMENT_ID}_PER_IMAGE.csv"
per_image.to_csv(per_image_path, index=False, encoding="utf-8")

print("Images                 :", len(per_image))
print("FP32 CIDEr-D           :", float(fp32_mean))
print("Mixed INT8 CIDEr-D     :", float(int8_mean))
print("Observed paired delta  :", observed_delta)
print("Exact caption agreement:",
      float(per_image["exact_caption_match"].mean()))

PTBTokenizer tokenized 74030 tokens at 219246.90 tokens per second.
PTBTokenizer tokenized 13429 tokens at 86960.06 tokens per second.
PTBTokenizer tokenized 13381 tokens at 89679.53 tokens per second.


Images                 : 1014
FP32 CIDEr-D           : 0.6145264982944407
Mixed INT8 CIDEr-D     : 0.6171285226440146
Observed paired delta  : 0.0026020243495739137
Exact caption agreement: 0.7564102564102564


## Cell 3 — 10,000-replicate paired bootstrap

In [3]:
BOOTSTRAP_SEED = 20261026
BOOTSTRAP_REPLICATES = 10000
NONINFERIORITY_MARGIN = -0.010
rng = np.random.default_rng(BOOTSTRAP_SEED)

# Lấy mẫu cùng chỉ số ảnh cho cả FP32 và INT8 để bảo toàn thiết kế paired.
bootstrap_means = np.empty(BOOTSTRAP_REPLICATES, dtype=np.float64)
batch_size = 500
for start in range(0, BOOTSTRAP_REPLICATES, batch_size):
    stop = min(start + batch_size, BOOTSTRAP_REPLICATES)
    indices = rng.integers(
        0,
        len(paired_delta),
        size=(stop - start, len(paired_delta)),
        dtype=np.int32,
    )
    bootstrap_means[start:stop] = paired_delta[indices].mean(axis=1)

ci95_lower, ci95_upper = np.quantile(bootstrap_means, [0.025, 0.975])
noninferiority_probability = float(
    np.mean(bootstrap_means > NONINFERIORITY_MARGIN)
)
positive_probability = float(np.mean(bootstrap_means > 0.0))
noninferiority_passed = bool(ci95_lower > NONINFERIORITY_MARGIN)
superiority_exploratory = bool(ci95_lower > 0.0)

bootstrap_summary = {
    "resampling_unit": "image_id",
    "paired_sampling": True,
    "replicates": BOOTSTRAP_REPLICATES,
    "random_seed": BOOTSTRAP_SEED,
    "observed_delta_CIDEr_D": observed_delta,
    "ci95_lower": float(ci95_lower),
    "ci95_upper": float(ci95_upper),
    "noninferiority_margin": NONINFERIORITY_MARGIN,
    "noninferiority_probability": noninferiority_probability,
    "positive_delta_probability": positive_probability,
    "noninferiority_passed": noninferiority_passed,
    "superiority_exploratory": superiority_exploratory,
}
assert noninferiority_passed, bootstrap_summary

np.save(
    WORK_ROOT / f"{EXPERIMENT_ID}_BOOTSTRAP_DELTAS.npy",
    bootstrap_means,
    allow_pickle=False,
)
print("Paired bootstrap       : COMPLETE")
print("Replicates             :", BOOTSTRAP_REPLICATES)
print("95% CI                 :", float(ci95_lower), "/", float(ci95_upper))
print("NI margin              :", NONINFERIORITY_MARGIN)
print("Non-inferiority        : PASSED")
print("Exploratory superiority:",
      "SUPPORTED" if superiority_exploratory else "NOT SUPPORTED")

Paired bootstrap       : COMPLETE
Replicates             : 10000
95% CI                 : -0.009995126278329478 / 0.014995039050558944
NI margin              : -0.01
Non-inferiority        : PASSED
Exploratory superiority: NOT SUPPORTED


## Cell 4 — Decision, limitations and next-phase authorization

In [4]:
unchanged = per_image[per_image["exact_caption_match"]]
changed = per_image[~per_image["exact_caption_match"]]
diagnostics = {
    "exact_caption_count": int(len(unchanged)),
    "changed_caption_count": int(len(changed)),
    "exact_caption_agreement": float(len(unchanged) / len(per_image)),
    "mean_delta_exact_subset": float(unchanged["delta_CIDEr_D"].mean()),
    "mean_delta_changed_subset": float(changed["delta_CIDEr_D"].mean()),
    "changed_subset_improved_count": int(
        (changed["delta_CIDEr_D"] > 0).sum()
    ),
    "changed_subset_degraded_count": int(
        (changed["delta_CIDEr_D"] < 0).sum()
    ),
    "changed_subset_tied_count": int(
        (changed["delta_CIDEr_D"] == 0).sum()
    ),
}

result = {
    "schema_version": "f30k_t3s3_int8_noninferiority_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "passed",
    "experiment_id": EXPERIMENT_ID,
    "source_experiment": VALIDATION_RESULT["experiment_id"],
    "source_result_sha256": sha256(VALIDATION_RESULT_PATH),
    "source_captions_sha256": sha256(CAPTIONS_PATH),
    "pair": VALIDATION_RESULT["pair"],
    "method": VALIDATION_RESULT["method"],
    "seed": VALIDATION_RESULT["seed"],
    "evaluation_split": "Flickr30k Karpathy Validation",
    "validation_images": 1014,
    "primary_endpoint": "paired per-image CIDEr-D delta (mixed INT8 - FP32)",
    "bootstrap": bootstrap_summary,
    "diagnostics": diagnostics,
    "original_strict_gate": {
        "name": "exact caption agreement",
        "required": original_gates["caption_agreement"]["minimum_required"],
        "observed": original_gates["caption_agreement"]["agreement"],
        "passed": False,
        "preserved_without_threshold_change": True,
    },
    "decision": (
        "MIXED_INT8_NONINFERIOR_ON_VALIDATION; "
        "VERBATIM_STABILITY_GATE_REMAINS_FAILED"
    ),
    "limitations": [
        "Exact-caption agreement is 75.64%, below the original 90% strict gate.",
        "Non-inferiority concerns CIDEr-D quality, not bitwise/verbatim stability.",
        "No Held-out Test data were read.",
    ],
    "method_or_threshold_changes_after_result": False,
    "held_out_test_authorized": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "next_phase_authorized": "F30K_T3S3_Deployment_Lock",
}
result_path = WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json"
write_json_atomic(result_path, result)

bootstrap_path = WORK_ROOT / f"{EXPERIMENT_ID}_BOOTSTRAP_DELTAS.npy"
manifest_files = [per_image_path, bootstrap_path, result_path]
manifest_path = WORK_ROOT / "MANIFEST.json"
write_json_atomic(manifest_path, {
    "schema_version": "f30k_t3s3_int8_noninferiority_manifest_v1",
    "experiment_id": EXPERIMENT_ID,
    "status": "complete",
    "artifacts": [
        {
            "relative_path": path.name,
            "size_bytes": path.stat().st_size,
            "sha256": sha256(path),
        }
        for path in manifest_files
    ],
    "test_set_status": "sealed",
    "test_set_used": False,
})

print("Status                 : PASSED")
print("Decision               :", result["decision"])
print("Original strict gate   : FAILED / PRESERVED")
print("Held-out Test          : SEALED / NOT USED")
print("Next                   :", result["next_phase_authorized"])
print("Output                 :", WORK_ROOT)

Status                 : PASSED
Decision               : MIXED_INT8_NONINFERIOR_ON_VALIDATION; VERBATIM_STABILITY_GATE_REMAINS_FAILED
Original strict gate   : FAILED / PRESERVED
Held-out Test          : SEALED / NOT USED
Next                   : F30K_T3S3_Deployment_Lock
Output                 : /kaggle/working/F30K_T3S3_INT8_NonInferiority
